In [1]:
from __future__ import annotations

from typing import Optional, Tuple
from arch import arch_model

import numpy as np
import pandas as pd
import torch
import json

from data_models import Scaler, Split, VolDataset

In [2]:
BATCH_SIZE = 64
EPS = 1e-8

In [3]:

def _garch_recursion(eps: np.ndarray, omega: float, alpha: float,
                     beta: float) -> np.ndarray:
    n = len(eps)
    s2 = np.empty(n)
    s2[0] = max(eps.var(), 1e-8)
    for t in range(1, n):
        s2[t] = omega + alpha * eps[t - 1] ** 2 + beta * s2[t - 1]
    return s2


def _garch_forward(returns: pd.Series, horizon: int, min_obs: int,
                   refit_every: int, verbose: bool) -> Tuple[pd.Series, int]:
    """Прогноз средней волатильности за дни t+1..t+h, сделанный на конец дня t.
    """

    r = (returns * 100.0).astype(float)          
    n = len(r)
    out = pd.Series(index=returns.index, dtype=float)

    params = None          # (mu, omega, alpha, beta)
    eps = sigma2 = None    
    last_fit = -10 ** 9
    fallbacks = 0
    days = 0

    for i in range(min_obs, n):
        days += 1
        refit = (params is None) or (i - last_fit >= refit_every)

        if refit:
            try:
                res = arch_model(r.iloc[:i + 1], vol='GARCH', p=1, q=1,
                                 dist='normal').fit(disp='off', show_warning=False)
                p = res.params
                params = (float(p['mu']), float(p['omega']),
                          float(p['alpha[1]']), float(p['beta[1]']))
                mu, omega, alpha, beta = params
                eps = r.iloc[:i + 1].values - mu
                sigma2 = _garch_recursion(eps, omega, alpha, beta)
                last_fit = i
            except Exception:
                params = None
        else:
            mu, omega, alpha, beta = params
            sigma2 = np.append(
                sigma2, omega + alpha * eps[-1] ** 2 + beta * sigma2[-1])
            eps = np.append(eps, r.iloc[i] - mu)

        if params is None:
            fallbacks += 1
            hist = r.iloc[max(0, i - 19):i + 1].std()
            out.iloc[i] = float(hist) / 100.0
            continue

        mu, omega, alpha, beta = params

        v = omega + alpha * eps[-1] ** 2 + beta * sigma2[-1]
        total, persist = v, alpha + beta

        for _ in range(horizon - 1):
            v = omega + persist * v
            total += v
        out.iloc[i] = float(np.sqrt(total / horizon)) / 100.0

        if verbose and (i - min_obs) % 500 == 0:
            print(f"    GARCH: {i}/{n}")

    return out, fallbacks

In [4]:
def _sanity_check(ds: VolDataset, verbose: bool = True) -> None:
    """Быстрые проверки, которые ловят большинство ошибок выравнивания."""
    h = ds.meta['horizon']
    problems = []

    for nm, sp in (('train', ds.train), ('val', ds.val), ('test', ds.test)):
        if len(sp) == 0:
            continue
        if not np.isfinite(sp.X).all():
            problems.append(f"{nm}: NaN/inf в X")
        if not np.isfinite(sp.y).all():
            problems.append(f"{nm}: NaN/inf в y")
        if not sp.dates.is_monotonic_increasing:
            problems.append(f"{nm}: даты не отсортированы")

    if len(ds.train) and len(ds.val):
        gap = (ds.val.dates[0] - ds.train.dates[-1]).days
        if ds.train.dates[-1] >= ds.val.dates[0]:
            problems.append("train заходит в val")
        elif verbose:
            print(f"  Зазор train->val: {gap} календарных дней "
                  f"(нужно >= {h} торговых)")
    if len(ds.val) and len(ds.test):
        if ds.val.dates[-1] >= ds.test.dates[0]:
            problems.append("val заходит в test")

    if len(ds.train) > 50:
        c = float(np.corrcoef(ds.train.X[:, -1], ds.train.y)[0, 1])
        if verbose:
            print(f"  corr(последний вход, таргет) = {c:.3f}")
        if c > 0.97:
            problems.append(f"подозрительно высокая корреляция {c:.3f} - "
                            f"похоже на пересечение окон")
    if problems:
        print("  !! ПРОБЛЕМЫ:")
        for p in problems:
            print(f"     - {p}")
    elif verbose:
        print("  Проверки пройдены.")



def build_dataset(
        returns: pd.Series,
        split_date: str,
        val_date: str,
        news_emb: Optional[pd.DataFrame] = None,
        window: int = 90,
        horizon: int = 5,
        past_window: int = 5,
        garch_min_obs: int = 500,
        garch_refit_every: int = 20,
        embargo: Optional[int] = None,
        garch_cache: Optional[str] = None,
        verbose: bool = True,
) -> VolDataset:
    """
    Параметры
    ---------
    returns       : pd.Series дневных доходностей
    split_date    : начало val (train - всё строго раньше).
    val_date      : начало test.
    news_emb      : DataFrame с DatetimeIndex, колонки - размерности эмбеддинга.
                    Берётся эмбеддинг дня t; пропуски заполняются последним
                    известным (ffill, только из прошлого). None -> нули.
    window        : длина окна для LSTM (дней).
    horizon       : горизонт прогноза h. y = RV за дни t+1..t+h.
    past_window   : окно сглаживания для признака RV (обычно = 5).
    garch_min_obs : минимум наблюдений до первого фита GARCH.
    garch_refit_every : раз во сколько дней переобучать GARCH.
    embargo       : сколько наблюдений выбросить на границе сплитов.
    garch_cache   : путь к .parquet для кеша прогнозов GARCH.
    """

    if embargo is None:
        embargo = horizon

    returns = returns.dropna().astype(float)
    if not isinstance(returns.index, pd.DatetimeIndex):
        raise TypeError("returns должен иметь DatetimeIndex")
    n = len(returns)
    r2 = returns ** 2

    rv_past = np.sqrt(r2.rolling(past_window).mean())

    rv_fwd = np.sqrt(r2.rolling(horizon).mean().shift(-horizon))

    # GARCH
    garch = None
    if garch_cache is not None:
        try:
            cached = pd.read_parquet(garch_cache)
            g = cached.iloc[:, 0]
            if g.index.equals(returns.index) and cached.attrs.get('h') == horizon:
                garch, fallbacks = g, int(cached.attrs.get('fallbacks', 0))
                if verbose:
                    print(f"  GARCH загружен из кеша: {garch_cache}")
        except Exception:
            garch = None

    if garch is None:
        if verbose:
            print(f"  Считаем GARCH(1,1), горизонт {horizon}д, "
                  f"перефит раз в {garch_refit_every}д...")
        garch, fallbacks = _garch_forward(
            returns, horizon, garch_min_obs, garch_refit_every, verbose)
        if garch_cache is not None:
            df = garch.to_frame('garch')
            df.attrs['h'] = horizon
            df.attrs['fallbacks'] = fallbacks
            df.to_parquet(garch_cache)

    # окно признаков
    first_t = max(window + past_window - 2, garch_min_obs)
    last_t = n - 1 - horizon
    if last_t <= first_t:
        raise ValueError("Слишком короткий ряд для заданных window/horizon.")

    log_rv = np.log(rv_past.values + EPS)
    windows = np.lib.stride_tricks.sliding_window_view(log_rv, window)

    t_idx = np.arange(first_t, last_t + 1)
    X_log = windows[t_idx - window + 1]                 # (N, window)
    y_log = np.log(rv_fwd.values[t_idx] + EPS)          # (N,)
    g_log = np.log(garch.values[t_idx] + EPS)           # (N,)
    y_raw = rv_fwd.values[t_idx]
    g_raw = garch.values[t_idx]
    dates = returns.index[t_idx]


    ok = (np.isfinite(X_log).all(axis=1) & np.isfinite(y_log)
          & np.isfinite(g_log))
    if not ok.all():
        dropped = int((~ok).sum())
        if verbose:
            print(f"  Отброшено наблюдений с NaN/inf: {dropped}")
        X_log, y_log, g_log = X_log[ok], y_log[ok], g_log[ok]
        y_raw, g_raw, dates = y_raw[ok], g_raw[ok], dates[ok]

    # новостные эмбеддинги
    if news_emb is not None:
        if not isinstance(news_emb.index, pd.DatetimeIndex):
            raise TypeError("news_emb должен иметь DatetimeIndex")
        aligned = news_emb.reindex(returns.index).ffill()
        emb_all = aligned.loc[dates].to_numpy(dtype=np.float32)
        emb_all = np.nan_to_num(emb_all, nan=0.0)
        emb_dim = emb_all.shape[1]
    else:
        emb_dim = 768
        emb_all = np.zeros((len(dates), emb_dim), dtype=np.float32)

    # сплит по датам
    sd, vd = pd.Timestamp(split_date), pd.Timestamp(val_date)
    masks = {
        'train': dates < sd,
        'val': (dates >= sd) & (dates < vd),
        'test': dates >= vd,
    }
    pos = {k: np.flatnonzero(m) for k, m in masks.items()}

    # таргет последнего train-примера
    if embargo > 0:
        for k in ('train', 'val'):
            if len(pos[k]) > embargo:
                pos[k] = pos[k][:-embargo]

    if len(pos['train']) == 0:
        raise ValueError("Пустой train - проверьте split_date и garch_min_obs.")

    # нормализация (статистики только по train)
    tr = pos['train']
    scaler = Scaler(
        x_mean=float(X_log[tr].mean()), x_std=float(X_log[tr].std() + EPS),
        y_mean=float(y_log[tr].mean()), y_std=float(y_log[tr].std() + EPS),
        g_mean=float(g_log[tr].mean()), g_std=float(g_log[tr].std() + EPS),
    )

    def make_split(idx: np.ndarray) -> Split:
        return Split(
            X=scaler.transform_x(X_log[idx]).astype(np.float32),
            y=scaler.transform_y(y_log[idx]).astype(np.float32),
            g=scaler.transform_g(g_log[idx]).astype(np.float32),
            emb=emb_all[idx],
            y_raw=y_raw[idx],
            g_raw=g_raw[idx],
            dates=dates[idx],
        )

    ds = VolDataset(
        train=make_split(pos['train']),
        val=make_split(pos['val']),
        test=make_split(pos['test']),
        scaler=scaler,
        meta={
            'window': window, 'horizon': horizon, 'past_window': past_window,
            'embargo': embargo, 'emb_dim': emb_dim,
            'garch_fallbacks': fallbacks, 'garch_days': n - garch_min_obs,
            'returns': returns,
        },
    )

    _sanity_check(ds, verbose=verbose)
    return ds

In [5]:
with open("../imoex_parser/data.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# Колонки MOEX
columns = [
    "BOARDID", "SECID", "TRADEDATE", "SHORTNAME", "NAME",
    "CLOSE", "OPEN", "HIGH", "LOW", "VALUE",
    "COL10", "COL11", "DECIMALS", "COL13", "CURRENCYID",
    "COL15", "COL16", "COL17", "ADMITTEDQUOTE", "COL19"
]

# DataFrame
df = pd.DataFrame(raw, columns=columns)
df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
df = df.set_index("TRADEDATE")

In [6]:
SPLIT_DATE = '2023-01-01'  # train: 2010–2023
VAL_DATE   = '2025-07-01'  # val:   2023-2025
# ну и test: 2025 - н.в

px = df['CLOSE'].astype(float).sort_index()
px = px[~px.index.duplicated(keep='last')]   # на всякий случай
returns = np.log(px).diff().dropna()          # лог-доходности

print(len(returns), returns.abs().median())
ds = build_dataset(returns, SPLIT_DATE, VAL_DATE)


4193 0.00664362706746946
  Считаем GARCH(1,1), горизонт 5д, перефит раз в 20д...
    GARCH: 500/4193
    GARCH: 1000/4193
    GARCH: 1500/4193
    GARCH: 2000/4193
    GARCH: 2500/4193
    GARCH: 3000/4193
    GARCH: 3500/4193
    GARCH: 4000/4193
  Зазор train->val: 11 календарных дней (нужно >= 5 торговых)
  corr(последний вход, таргет) = 0.478
  Проверки пройдены.


In [7]:
# torch.save(ds, 'train.data')